# 🌍 Good Vibes Workshop — ECCB 2026

## Responsible use of AI for large-scale, rapid Earth Observation workflows

**Authors:** AB, HFT, HH, JW  
**Affiliation:** Institute of Zoology, Zoological Society of London  
**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

---

### What you'll do

You will use **an AI assistant** (Gemini, ChatGPT, Copilot — your choice) to write Earth Observation analysis code. You will not be given working code. You will prompt it into existence, run it, fix it when it breaks, and build complexity until you're doing real analysis.

**Open your AI assistant in another browser tab now.**

### How this notebook is structured

- Early on, we give you **detailed prompt guidance** — almost word-for-word what to ask
- As you progress, the guidance gets **vaguer** — just a goal and some hints
- By the end, you're on your own

The only pre-written code you'll see is either:
- ⚙️ **Setup code** (authentication, imports)
- 🐛 **Buggy code** that you need to fix

---
# Section 0: Setup ⚙️

This is the only section with pre-written working code. Run all four cells.

In [ ]:
!pip install -q geemap earthengine-api

In [ ]:
import ee
import geemap
import datetime
import sys
import matplotlib.pyplot as plt

In [ ]:
ee.Authenticate()

In [ ]:
# ⚡ Replace 'YOUR-PROJECT-ID' with your Google Cloud Project ID
ee.Initialize(project='YOUR-PROJECT-ID')

# Quick test
print(f'✅ Ready! ee version {ee.__version__}, geemap version {geemap.__version__}')
print(f'📅 {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}, Python {sys.version.split(" ")[0]}')

---
# Worked Example 1: Visualise Satellite Imagery 🛰️

### 🎯 Goal
Display satellite imagery of **Sabah, Borneo** on an interactive map.

---

### Task 1A: True-colour satellite image

Copy this prompt into your AI assistant:

> *Write Python code for Google Colab that uses the `earthengine-api` and `geemap` packages to display a true-colour Landsat 8 surface reflectance composite of Sabah, Borneo (approximately 115.5–118.5°E, 4.5–7°N) for 2023. Filter for images with less than 30% cloud cover, take the median composite, and display it on an interactive geemap map. The Landsat 8 Collection 2 Surface Reflectance true-colour bands are SR_B4, SR_B3, SR_B2.*

Paste the AI's output below and run it.

In [ ]:
# ✍️ Task 1A: Paste your AI-generated code here



**Did it work?** You should see an interactive map with satellite imagery of Borneo. If it failed, give your AI the error message and ask it to fix the code.

> ⚠️ **Teaching Point:** We gave the AI very specific information — the exact package names, the dataset, the band names, the coordinates. This is **Principle 1: Treat AI as an intelligence** — the more context you provide, the better the output.

---

### Task 1B: Add more layers

Now give your AI this follow-up:

> *Add a false-colour composite layer using bands SR_B5, SR_B4, SR_B3 (NIR, Red, Green) to highlight vegetation. Also load protected area boundaries from the WDPA dataset (`ee.FeatureCollection('WCMC/WDPA/current/polygons')`) filtered to the Sabah region and add them to the map.*

Paste below and run.

In [ ]:
# ✍️ Task 1B: Paste your AI-generated code here



**Toggle the layers** on the map. In the false-colour view, bright red/pink = dense vegetation. Where can you see the contrast between forest and cleared land?

> ⚠️ **Teaching Point:** You just built a multi-layer satellite visualisation in two prompts. **Iterative prompting** — building on previous output — is the core technique of vibe coding. You don't need to write the whole thing at once.

---

---
# Worked Example 2: Deforestation Analysis 🌳➡️🏜️

### 🎯 Goal
Analyse annual deforestation in Sabah using the Hansen Global Forest Change dataset. Calculate loss area by year and plot the results.

This is harder. The AI will probably get some things wrong. That's the point.

---

### Task 2A: Generate the analysis code

Ask your AI to analyse deforestation. Here's a prompt — but notice we're being **less specific** than before. We're deliberately leaving out some details to see what happens:

> *Write Python code using earthengine-api to analyse deforestation in Sabah, Borneo (115.5–118.5°E, 4.5–7°N) using the Hansen Global Forest Change dataset. Calculate the total area of forest lost each year from 2001 to 2023 in hectares, and plot the results as a bar chart with matplotlib.*

Paste the AI's output below and run it.

In [ ]:
# ✍️ Task 2A: Paste your AI-generated code here



### What happened?

**If it worked first time** — check the results carefully. Do the numbers seem plausible? (Sabah loses roughly 50,000–150,000 hectares of forest per year.)

**If it failed** — common reasons at this point:

| Error you might see | What went wrong | How to fix |
|---|---|---|
| `Pattern 'tree_cover' did not match any bands` | AI guessed the wrong band name | The correct band names are: `treecover2000`, `loss`, `gain`, `lossyear`, `datamask` |
| `Computation timed out` | AI used too fine a scale or too complex a computation | Add `bestEffort=True` to `reduceRegion()` |
| Wrong dataset ID | AI used an old or made-up dataset ID | The correct ID is `UMD/hansen/global_forest_change_2023_v1_11` |

Give the AI the error message and the correct information from the table above. Paste the fixed version below.

> ⚠️ **Teaching Point — Principle 2: Distrust.** We deliberately gave a less specific prompt and the AI likely got band names or dataset details wrong. AI is **confidently incorrect** about dataset-specific details. Always check against official documentation.

In [ ]:
# ✍️ Task 2A (fixed): Paste your corrected code here if the first attempt failed



### Task 2B: Visualise on a map

Now ask your AI to create a map. You decide what to include — but you probably want tree cover, forest loss, and maybe loss coloured by year.

> 💡 **Hint:** You might want to mention `geemap`, the study area, and what layers to show. Think about what colour palette makes sense for deforestation data.

In [ ]:
# ✍️ Task 2B: Paste your AI-generated map code here



---

---
# Worked Example 3: Find the Silent Bugs 🔇

### 🎯 Goal

The code below was generated by AI. It runs **without errors** and produces numbers. But **three of the results are wrong.**

Work in pairs. Can you find all three bugs?

---

### 🐛 Buggy code — runs but gives wrong answers

In [ ]:
# 🐛 THIS CODE HAS 3 SILENT BUGS — find them!
# It runs without errors, but the results are scientifically WRONG.

hansen = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
sabah = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# --- Statistic 1: Average tree cover in 2000 ---
tree_cover = hansen.select('treecover2000')
avg_tc = tree_cover.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sabah,
    scale=1000,
    maxPixels=1e13,
    bestEffort=True
).getInfo()
print(f'Average tree cover (2000): {avg_tc["treecover2000"]:.1f}%')

# --- Statistic 2: Total forest loss 2010-2020 ---
loss_year = hansen.select('lossyear')
loss_mask = loss_year.gte(10).And(loss_year.lte(19))
loss_area = loss_mask.multiply(ee.Image.pixelArea()).divide(10000)
total_loss = loss_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sabah,
    scale=30,
    maxPixels=1e13,
    bestEffort=True
).getInfo()
print(f'Forest loss 2010-2020: {total_loss["lossyear"]:,.0f} hectares')

# --- Statistic 3: Loss as percentage of original forest ---
forest_mask = tree_cover.gte(30)
forest_area = forest_mask.multiply(ee.Image.pixelArea()).divide(10000)
total_forest = forest_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sabah,
    scale=30,
    maxPixels=1e13,
    bestEffort=True
).getInfo()

pct_lost = (total_loss['lossyear'] / total_forest['treecover2000']) * 100
print(f'Original forest area: {total_forest["treecover2000"]:,.0f} hectares')
print(f'Percentage lost (2010-2020): {pct_lost:.1f}%')

### 🔍 Bug hunt

**Discuss with a partner.** Three things are wrong. Think about:
- What does each reducer actually compute? Is it the right one for this statistic?
- What is the native resolution of this dataset? What happens if you use a different scale?
- What year range are we actually capturing? Count carefully.

When you think you've found them all, open the hints below.

In [ ]:
# @title 👁️ Click to reveal the 3 bugs

print("""BUG #1: ee.Reducer.sum() instead of ee.Reducer.mean()
  → Summing percentages across millions of pixels gives a
    nonsensical number. We want the AVERAGE tree cover.
  → Fix: ee.Reducer.mean()

BUG #2: scale=1000 instead of scale=30
  → Hansen data is 30m resolution. Using 1000m means GEE
    averages ~1,111 pixels into one, destroying fine detail.
  → Fix: scale=30

BUG #3: .lte(19) instead of .lte(20)
  → lossyear=20 means year 2020. The code only captures
    2010–2019, missing 2020 entirely. Off-by-one error.
  → Fix: .lte(20)

All three are bugs that real AI systems commonly generate.
""")

### Now fix it

You can either fix the code manually or ask your AI: *"This code has three bugs: [describe them]. Fix it."*

Paste your corrected version below and verify the results make sense.

In [ ]:
# ✍️ Paste your fixed version here



> ⚠️ **Teaching Point — Principle 3: Verify.** Silent bugs are the most dangerous kind. The code ran. It produced numbers. But the numbers were wrong. Always ask: *"Is this result the right order of magnitude?"* and *"Can I check this against a known value?"*

---

---
# Exercise 1: Your Region, Your Prompts ✍️

### 🎯 Goal
Analyse deforestation in **a region you choose**.

---

You've seen how to prompt, how to debug, and how to verify. Now do it yourself.

1. **Pick a region** (or use one of these):
   - Brazilian Amazon: `[-62, -5, -58, -1]`
   - Congo Basin: `[18, -2, 22, 2]`
   - Sumatra: `[101, -2, 105, 2]`
   - Madagascar: `[46, -17, 50, -13]`

2. **Write your own prompt.** Think about what information the AI needs to get it right. What did you learn from Worked Examples 1 and 2 about what to include?

3. **Run the code, debug, verify.** Are the numbers plausible? Can you check them?

> 💡 The less you tell the AI, the more likely it is to get dataset-specific details wrong. But being too prescriptive defeats the purpose. Find the balance.

In [ ]:
# ✍️ Exercise 1: Your code here



In [ ]:
# ✍️ Exercise 1 (continued)



---
# Exercise 2: Trend Challenge 🏆

### 🎯 Challenge

**What is the 5-year period with the highest average annual deforestation in Sabah?**

Use the Hansen dataset. Study area: `[115.5, 4.5, 118.5, 7.0]`. Closest answer wins a prize.

No guidance. No template. Vibe code it.

In [ ]:
# 🏆 Your challenge code



In [ ]:
# 🏆 More space



---
# Exercise 3: Innovate 🚀

*If you've finished the challenge or want to explore.*

Build something we haven't shown you. Some dataset IDs to get you started:

- Fire hotspots: `FIRMS`
- Vegetation index: `MODIS/061/MOD13A1`
- Surface water: `JRC/GSW1_4/GlobalSurfaceWater`
- Land surface temperature: `MODIS/061/MOD11A2`
- Nighttime lights: `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG`

Remember the six principles.

In [ ]:
# 🚀 Your innovation



---
# Wrap-up 🎓

### What you practised

| Principle | Where |
|-----------|-------|
| **1. Treat AI as an intelligence** | WE1 — giving specific context in prompts |
| **2. Distrust** | WE2 — AI got band names wrong |
| **3. Verify** | WE3 — finding silent bugs in running code |
| **4. Prioritize reporting** | All — understanding code before trusting results |
| **5. Engineer adversity** | WE3 — checking if results are the right order of magnitude |
| **6. Maximise transparency** | Setup — capturing reproducibility metadata |

### Resources

- [GEE Dataset Catalog](https://developers.google.com/earth-engine/datasets) — always check band names here
- [geemap docs](https://geemap.org/)
- [Hansen Global Forest Change](https://glad.earthengine.app/view/global-forest-change)

📋 **Print the vibe coding checklist** and stick it on your monitor.  
💾 **Save this notebook:** File → Save a copy in Drive

---

*Good Vibes Workshop — ECCB 2026 | AB, HFT, HH, JW | ZSL Institute of Zoology | CC BY 4.0*